## **Dependencies**

In [1]:
!pip install -q -U datasets rank-bm25 tqdm nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 48.7 MB/s eta 0:00:00:00:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
moviepy 1.0.3 requires decorator<5.0,>=4.0.2, but you have decorator 5.3.1 which is incompatible.


## **Imports**

In [2]:
import json
import os
import pickle
import re
from typing import Dict, List, Set, Tuple
from tqdm import tqdm
from rank_bm25 import BM25Okapi
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

In [4]:
nltk.pathsec.ALLOW_PROXIED_FETCH = True
nltk.download('stopwords', quiet=True)

True

## **Configuration Paths**

In [5]:
DATA_DIR = "/kaggle/input/datasets/anarvaaa/original-scifact-data"
CORPUS_PATH = os.path.join(DATA_DIR, "corpus.jsonl")
TRAIN_CLAIMS_PATH = os.path.join(DATA_DIR, "claims_train.jsonl")
OUTPUT_DIR = "/kaggle/working/scifact_bm25_index"

os.makedirs(OUTPUT_DIR, exist_ok=True)

## **Preprocessing**

In [6]:
stop_words = set(stopwords.words("english"))
stemmer = PorterStemmer()

def tokenize(text: str, remove_stopwords: bool = True, use_stemming: bool = True) -> List[str]:
    """
    Clean, tokenize, remove stopwords, and stem tokens for robust scientific BM25 retrieval.
    """
    # Lowercase and extract alphanumeric terms (including hyphenated scientific terms)
    tokens = re.findall(r"\b[a-zA-Z0-9]+(?:-[a-zA-Z0-9]+)*\b", text.lower())
    
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stop_words]
    if use_stemming:
        tokens = [stemmer.stem(t) for t in tokens]
        
    return tokens

## **Building Documents**

In [7]:
print(f"Loading corpus from: {CORPUS_PATH}")

corpus_docs = []      # Raw combined text (title + abstract)
doc_ids = []          # Maps list index -> scifact doc_id (int)
doc_id_to_idx = {}    # Maps scifact doc_id -> list index
doc_metadata = {}     # Maps doc_id -> {title, abstract, sentences}

with open(CORPUS_PATH, "r", encoding="utf-8") as f:
    for line in tqdm(f, desc="Reading corpus.jsonl"):
        item = json.loads(line.strip())
        doc_id = item["doc_id"]
        title = item.get("title", "")
        
        # SciFact abstract is a list of sentence strings
        abstract_sentences = item.get("abstract", [])
        if isinstance(abstract_sentences, list):
            abstract_text = " ".join(abstract_sentences)
        else:
            abstract_text = str(abstract_sentences)
            abstract_sentences = [abstract_text]

        # Combine title and abstract
        full_text = f"{title} {abstract_text}".strip()
        
        idx = len(doc_ids)
        doc_ids.append(doc_id)
        doc_id_to_idx[doc_id] = idx
        corpus_docs.append(full_text)
        
        doc_metadata[doc_id] = {
            "doc_id": doc_id,
            "title": title,
            "abstract_text": abstract_text,
            "abstract_sentences": abstract_sentences
        }

print(f"Total documents loaded: {len(doc_ids):,}")

# Tokenize full corpus
print("Tokenizing corpus...")
tokenized_corpus = [tokenize(doc) for doc in tqdm(corpus_docs, desc="Tokenizing")]

Loading corpus from: /kaggle/input/datasets/anarvaaa/original-scifact-data/corpus.jsonl


Reading corpus.jsonl: 5183it [00:00, 40604.80it/s]


Total documents loaded: 5,183
Tokenizing corpus...


Tokenizing: 100%|██████████| 5183/5183 [00:14<00:00, 367.16it/s]


## **BM25 Params Tuning (Optional)**

In [10]:

def evaluate_bm25_params(
    train_path: str,
    tokenized_corpus: List[List[str]],
    doc_ids: List[int],
    k1_list: List[float] = [1.2, 1.5, 1.8],
    b_list: List[float] = [0.5, 0.75, 0.85],
    k_eval: int = 20
) -> Tuple[float, float, float]:
    """
    Evaluates Recall@k and MRR@k across candidate (k1, b) values using claims_train.jsonl.
    """
    # Load training claims with valid cited documents
    train_queries = []
    with open(train_path, "r", encoding="utf-8") as f:
        for line in f:
            data = json.loads(line.strip())
            # Use cited_doc_ids as ground truth gold documents
            gold_docs = set(data.get("cited_doc_ids", []))
            if gold_docs:
                train_queries.append({
                    "id": data["id"],
                    "claim": data["claim"],
                    "gold_docs": gold_docs
                })

    print(f"\nTuning BM25 on {len(train_queries)} training claims (evaluating Recall@{k_eval}, MRR@{k_eval})...")

    best_score = -1.0
    best_params = (1.5, 0.75)

    for k1 in k1_list:
        for b in b_list:
            bm25 = BM25Okapi(tokenized_corpus, k1=k1, b=b)
            
            recalls = []
            mrrs = []
            
            for q in train_queries:
                q_tokens = tokenize(q["claim"])
                scores = bm25.get_scores(q_tokens)
                
                # Get top-k indices
                top_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k_eval]
                retrieved_doc_ids = [doc_ids[i] for i in top_indices]
                
                # Recall@k
                hits = set(retrieved_doc_ids).intersection(q["gold_docs"])
                recall = len(hits) / len(q["gold_docs"])
                recalls.append(recall)
                
                # MRR@k
                mrr = 0.0
                for rank, d_id in enumerate(retrieved_doc_ids, start=1):
                    if d_id in q["gold_docs"]:
                        mrr = 1.0 / rank
                        break
                mrrs.append(mrr)
                
            mean_recall = sum(recalls) / len(recalls)
            mean_mrr = sum(mrrs) / len(mrrs)
            
            print(f"k1={k1:.2f}, b={b:.2f} -> Recall@{k_eval}: {mean_recall:.4f}, MRR@{k_eval}: {mean_mrr:.4f}")
            
            if mean_recall > best_score:
                best_score = mean_recall
                best_params = (k1, b)

    print(f"\nOptimal parameters: k1={best_params[0]}, b={best_params[1]} (Best Recall@{k_eval}: {best_score:.4f})")
    return best_params

# Run tuning on train set (set to False if you want default k1=1.5, b=0.75)
RUN_TUNING = False
if RUN_TUNING and os.path.exists(TRAIN_CLAIMS_PATH):
    best_k1, best_b = evaluate_bm25_params(TRAIN_CLAIMS_PATH, tokenized_corpus, doc_ids, k_eval=20)
else:
    best_k1, best_b = 1.5, 0.75


## **Build Index**

In [11]:
print(f"\nBuilding final BM25 index with k1={best_k1}, b={best_b}...")
final_bm25 = BM25Okapi(tokenized_corpus, k1=best_k1, b=best_b)
print("Completed")


Building final BM25 index with k1=1.5, b=0.75...
Completed


## **Save Artifacts**

In [12]:
# ==========================================
# 7. Serialize Artifacts to /kaggle/working/
# ==========================================
artifacts = {
    "bm25_model": final_bm25,
    "doc_ids": doc_ids,                  # list of doc_id by index
    "doc_id_to_idx": doc_id_to_idx,      # doc_id -> index
    "doc_metadata": doc_metadata,        # doc_id -> title, abstract, sentences
    "params": {"k1": best_k1, "b": best_b, "stemmed": True, "stopword_removed": True}
}

index_file_path = os.path.join(OUTPUT_DIR, "scifact_bm25_index.pkl")
with open(index_file_path, "wb") as f:
    pickle.dump(artifacts, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved BM25 artifacts successfully to: {index_file_path}")
print(f"File size: {os.path.getsize(index_file_path) / (1024 * 1024):.2f} MB")

Saved BM25 artifacts successfully to: /kaggle/working/scifact_bm25_index/scifact_bm25_index.pkl
File size: 20.21 MB


## **Verification**

In [13]:
# ==========================================
# 8. Quick Verification Search Test
# ==========================================
test_query = "Activation of PPM1D suppresses p53 function."
query_tokens = tokenize(test_query)
scores = final_bm25.get_scores(query_tokens)
top20_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:5]

print("\n--- Test Retrieval (Top 5) ---")
print(f"Query: '{test_query}'")
for rank, idx in enumerate(top20_indices, start=1):
    d_id = doc_ids[idx]
    meta = doc_metadata[d_id]
    print(f"Rank {rank} | doc_id: {d_id} | BM25 Score: {scores[idx]:.4f} | Title: {meta['title'][:60]}...")


--- Test Retrieval (Top 5) ---
Query: 'Activation of PPM1D suppresses p53 function.'
Rank 1 | doc_id: 5956380 | BM25 Score: 24.4409 | Title: Exome sequencing identifies somatic gain-of-function PPM1D m...
Rank 2 | doc_id: 4414547 | BM25 Score: 20.5951 | Title: Mosaic PPM1D mutations are associated with predisposition to...
Rank 3 | doc_id: 9483851 | BM25 Score: 14.4232 | Title: Unravelling mechanisms of p53-mediated tumour suppression...
Rank 4 | doc_id: 42489926 | BM25 Score: 14.2946 | Title: Small-molecule antagonists of p53-MDM2 binding: research too...
Rank 5 | doc_id: 21307488 | BM25 Score: 14.2551 | Title: HER-2/neu induces p53 ubiquitination via Akt-mediated MDM2 p...
